# House Prices: PyTorch 선형 baseline

## 요약

- 모델: `nn.Linear(input_dim, 1)`, 은닉층·활성화 함수·L2 없음
- 5-fold RMSLE: **0.144820 ± 0.038047**
- OOF RMSLE: **0.148764**
- Kaggle public RMSLE: **0.13990**
- 모든 전처리는 각 fold의 학습 데이터에서만 적합
- test **1,459행** 예측과 제출 형식 검증 완료

## 설정

- 모델: `nn.Module` 안의 `nn.Linear(input_dim, 1)` 하나
- 타깃: `log1p(SalePrice)`
- 손실: MSE
- optimizer: Adam, L2/weight decay 없음
- 검증: `KFold(5, shuffle=True, random_state=42)`

전처리는 각 fold의 학습 데이터에서만 적합한다.

In [1]:
from __future__ import annotations

import hashlib
import json
import platform
import random
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import torch
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_log_error
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from torch import nn

ROOT = Path.cwd()
if not (ROOT / "data" / "train.csv").exists():
    ROOT = ROOT.parent

TRAIN_PATH = ROOT / "data" / "train.csv"
TEST_PATH = ROOT / "data" / "test.csv"
SAMPLE_SUBMISSION_PATH = ROOT / "data" / "sample_submission.csv"
METRICS_PATH = ROOT / "reports" / "baseline_metrics.json"
OOF_PATH = ROOT / "reports" / "baseline_oof_predictions.csv"
TEST_PREDICTIONS_PATH = ROOT / "reports" / "baseline_test_fold_predictions.csv"
SUBMISSION_VALIDATION_PATH = ROOT / "reports" / "baseline_submission_validation.json"

EXPERIMENT_ID = "BASE-20260719-TORCH-LINEAR-SUB-01"
SUBMISSION_PATH = ROOT / "submissions" / f"submission_{EXPERIMENT_ID}.csv"
KAGGLE_PUBLIC_SCORE = 0.13990
KAGGLE_SCORE_RECORDED_AT_UTC = "2026-07-19T11:06:59Z"
SEED = 42
N_SPLITS = 5
LEARNING_RATE = 0.01
MAX_EPOCHS = 2000
PATIENCE = 150
MIN_DELTA = 1e-6
DEVICE = torch.device("cpu")

NUMERIC_COLUMNS = [
    "LotFrontage", "LotArea", "OverallQual", "OverallCond", "YearBuilt",
    "YearRemodAdd", "MasVnrArea", "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF",
    "TotalBsmtSF", "1stFlrSF", "2ndFlrSF", "LowQualFinSF", "GrLivArea",
    "BsmtFullBath", "BsmtHalfBath", "FullBath", "HalfBath", "BedroomAbvGr",
    "KitchenAbvGr", "TotRmsAbvGrd", "Fireplaces", "GarageYrBlt", "GarageCars",
    "GarageArea", "WoodDeckSF", "OpenPorchSF", "EnclosedPorch", "3SsnPorch",
    "ScreenPorch", "PoolArea", "MiscVal", "YrSold",
]

train = pd.read_csv(TRAIN_PATH, keep_default_na=False)
test = pd.read_csv(TEST_PATH, keep_default_na=False)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)
CATEGORICAL_COLUMNS = [
    column
    for column in train.columns
    if column not in {*NUMERIC_COLUMNS, "Id", "SalePrice"}
]

assert train.shape == (1460, 81)
assert test.shape == (1459, 80)
assert list(test.columns) == [column for column in train.columns if column != "SalePrice"]
assert list(sample_submission.columns) == ["Id", "SalePrice"]
assert len(sample_submission) == len(test)
assert sample_submission["Id"].equals(test["Id"])
assert str(sample_submission["Id"].dtype) == "int64"
assert str(sample_submission["SalePrice"].dtype) == "float64"
assert len(NUMERIC_COLUMNS) == 34
assert len(CATEGORICAL_COLUMNS) == 45
assert train["Id"].is_unique and train["SalePrice"].gt(0).all()

source_sha256 = hashlib.sha256(TRAIN_PATH.read_bytes()).hexdigest()
test_sha256 = hashlib.sha256(TEST_PATH.read_bytes()).hexdigest()
sample_submission_sha256 = hashlib.sha256(
    SAMPLE_SUBMISSION_PATH.read_bytes()
).hexdigest()
y = train["SalePrice"].to_numpy(dtype=np.float64)
y_log = np.log1p(y)

## 전처리

시설이 없는 경우는 전용 범주나 0으로 표시한다. 나머지 수치 결측은 중앙값,
범주형 결측은 `Unknown`으로 처리하고 one-hot encoding을 적용한다.

In [2]:
BASEMENT_COLUMNS = [
    "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2"
]
GARAGE_COLUMNS = ["GarageType", "GarageFinish", "GarageQual", "GarageCond"]


def clean_rows(frame):
    cleaned = frame.copy()

    for column in NUMERIC_COLUMNS:
        cleaned[column] = pd.to_numeric(
            cleaned[column].replace({"NA": np.nan, "": np.nan}),
            errors="coerce",
        )

    cleaned.loc[cleaned["Id"].eq(524), "YearRemodAdd"] = 2007

    basement_absent = cleaned["TotalBsmtSF"].fillna(0).eq(0)
    garage_absent = (
        cleaned["GarageCars"].fillna(0).eq(0)
        & cleaned["GarageArea"].fillna(0).eq(0)
    )

    for column in BASEMENT_COLUMNS:
        missing = cleaned[column].isin(["NA", ""])
        cleaned.loc[missing & basement_absent, column] = "NoBasement"
        cleaned.loc[missing & ~basement_absent, column] = "Unknown"

    for column in GARAGE_COLUMNS:
        missing = cleaned[column].isin(["NA", ""])
        cleaned.loc[missing & garage_absent, column] = "NoGarage"
        cleaned.loc[missing & ~garage_absent, column] = "Unknown"
    cleaned.loc[garage_absent, "GarageYrBlt"] = 0.0

    fireplace_absent = cleaned["Fireplaces"].fillna(0).eq(0)
    pool_absent = cleaned["PoolArea"].fillna(0).eq(0)
    cleaned.loc[
        cleaned["FireplaceQu"].isin(["NA", ""]) & fireplace_absent,
        "FireplaceQu",
    ] = "NoFireplace"
    cleaned.loc[
        cleaned["PoolQC"].isin(["NA", ""]) & pool_absent,
        "PoolQC",
    ] = "NoPool"

    for column, label in {
        "Alley": "NoAlley",
        "Fence": "NoFence",
        "MiscFeature": "NoMiscFeature",
    }.items():
        cleaned[column] = cleaned[column].replace({"NA": label, "": label})

    cleaned["MSSubClass"] = cleaned["MSSubClass"].map(lambda value: f"SC{value}")
    cleaned["MoSold"] = cleaned["MoSold"].map(lambda value: f"M{int(value):02d}")

    for column in CATEGORICAL_COLUMNS:
        cleaned[column] = cleaned[column].replace({"NA": "Unknown", "": "Unknown"})
    return cleaned


X = clean_rows(train.drop(columns="SalePrice"))
X_test = clean_rows(test)


def make_preprocessor():
    numeric = Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )
    categorical = Pipeline(
        [
            ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
            (
                "onehot",
                OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            ),
        ]
    )
    return ColumnTransformer(
        [
            ("numeric", numeric, NUMERIC_COLUMNS),
            ("categorical", categorical, CATEGORICAL_COLUMNS),
        ],
        remainder="drop",
        sparse_threshold=0.0,
    )

## 모델과 학습

활성화 함수나 은닉층이 없으므로 이 모델은 선형회귀다. 검증 점수가 가장 좋았던
가중치를 복원하며, optimizer의 `weight_decay`는 0이다.

In [3]:
class LinearHousePriceModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.linear = nn.Linear(input_dim, 1)

    def forward(self, x):
        return self.linear(x).squeeze(1)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def train_fold(X_train, y_train, X_valid, y_valid, seed):
    set_seed(seed)
    X_train_tensor = torch.as_tensor(X_train, dtype=torch.float32, device=DEVICE)
    y_train_tensor = torch.as_tensor(y_train, dtype=torch.float32, device=DEVICE)
    X_valid_tensor = torch.as_tensor(X_valid, dtype=torch.float32, device=DEVICE)
    y_valid_tensor = torch.as_tensor(y_valid, dtype=torch.float32, device=DEVICE)

    model = LinearHousePriceModel(X_train.shape[1]).to(DEVICE)
    nn.init.zeros_(model.linear.weight)
    nn.init.constant_(model.linear.bias, float(y_train.mean()))

    loss_fn = nn.MSELoss()
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=0.0,
    )

    best_score = float("inf")
    best_epoch = 0
    best_state = None
    epochs_without_improvement = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        optimizer.zero_grad()
        train_prediction = model(X_train_tensor)
        train_loss = loss_fn(train_prediction, y_train_tensor)
        train_loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            valid_prediction = model(X_valid_tensor)
            valid_loss = loss_fn(valid_prediction, y_valid_tensor)
            valid_rmsle = float(torch.sqrt(valid_loss).cpu())

        if valid_rmsle < best_score - MIN_DELTA:
            best_score = valid_rmsle
            best_epoch = epoch
            best_state = {
                name: value.detach().cpu().clone()
                for name, value in model.state_dict().items()
            }
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= PATIENCE:
            break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        best_prediction = model(X_valid_tensor).cpu().numpy().astype(np.float64)

    return model, best_prediction, {
        "best_epoch": best_epoch,
        "stopping_epoch": epoch,
        "best_validation_rmsle": best_score,
        "final_train_loss": float(train_loss.detach().cpu()),
        "final_validation_loss": float(valid_loss.detach().cpu()),
    }

## 5-fold 결과

각 fold에서 새 전처리기와 새 모델을 학습한다. validation 예측은 OOF 점수에 사용하고,
같은 모델의 test 로그 예측 5개는 평균해 제출 가격으로 변환한다.

In [4]:
kfold = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
oof_log_prediction = np.full(len(X), np.nan, dtype=np.float64)
test_log_predictions = []
fold_rows = []

for fold, (train_index, valid_index) in enumerate(kfold.split(X), start=1):
    preprocessor = make_preprocessor()
    X_train = preprocessor.fit_transform(X.iloc[train_index]).astype(np.float32)
    X_valid = preprocessor.transform(X.iloc[valid_index]).astype(np.float32)
    X_test_fold = preprocessor.transform(X_test).astype(np.float32)

    model, prediction, training = train_fold(
        X_train,
        y_log[train_index],
        X_valid,
        y_log[valid_index],
        seed=SEED + fold,
    )
    oof_log_prediction[valid_index] = prediction
    model.eval()
    with torch.no_grad():
        test_prediction = model(
            torch.as_tensor(X_test_fold, dtype=torch.float32, device=DEVICE)
        ).cpu().numpy().astype(np.float64)
    test_log_predictions.append(test_prediction)

    fold_rmsle = float(
        np.sqrt(np.mean((y_log[valid_index] - prediction) ** 2))
    )
    fold_rows.append(
        {
            "fold": fold,
            "train_rows": len(train_index),
            "validation_rows": len(valid_index),
            "encoded_features": X_train.shape[1],
            "best_epoch": training["best_epoch"],
            "stopping_epoch": training["stopping_epoch"],
            "rmsle": fold_rmsle,
        }
    )

fold_results = pd.DataFrame(fold_rows)
predicted_price = np.clip(np.expm1(oof_log_prediction), 0, None)
oof_rmsle = float(np.sqrt(mean_squared_log_error(y, predicted_price)))
cv_mean = float(fold_results["rmsle"].mean())
cv_std = float(fold_results["rmsle"].std(ddof=1))

test_log_prediction_matrix = np.vstack(test_log_predictions)
test_log_prediction_mean = test_log_prediction_matrix.mean(axis=0)
test_price_prediction = np.clip(np.expm1(test_log_prediction_mean), 0, None)

oof_frame = pd.DataFrame(
    {
        "Id": train["Id"],
        "SalePrice": y,
        "oof_log_prediction": oof_log_prediction,
        "oof_saleprice_prediction": predicted_price,
    }
)
oof_frame.to_csv(OOF_PATH, index=False)

test_prediction_frame = pd.DataFrame(
    {
        "Id": test["Id"],
        **{
            f"fold_{fold}_log_prediction": test_log_prediction_matrix[fold - 1]
            for fold in range(1, N_SPLITS + 1)
        },
        "mean_log_prediction": test_log_prediction_mean,
        "SalePrice": test_price_prediction,
    }
)
test_prediction_frame.to_csv(TEST_PREDICTIONS_PATH, index=False)

submission = sample_submission.copy()
submission["SalePrice"] = test_price_prediction
assert list(submission.columns) == list(sample_submission.columns)
assert len(submission) == len(test)
assert submission["Id"].equals(test["Id"])
assert submission["Id"].is_unique
assert np.isfinite(submission["SalePrice"]).all()
assert submission["SalePrice"].gt(0).all()
assert str(submission["Id"].dtype) == "int64"
assert str(submission["SalePrice"].dtype) == "float64"
SUBMISSION_PATH.parent.mkdir(parents=True, exist_ok=True)
submission.to_csv(SUBMISSION_PATH, index=False)

display(fold_results.round(6))
print(f"CV RMSLE: {cv_mean:.6f} ± {cv_std:.6f}")
print(f"OOF RMSLE: {oof_rmsle:.6f}")
print(f"Submission: {SUBMISSION_PATH.relative_to(ROOT)}")

,fold,train_rows,validation_rows,encoded_features,best_epoch,stopping_epoch,rmsle
0,1,1168,292,327,1234,1384,0.127152
1,2,1168,292,327,625,775,0.126484
2,3,1168,292,323,13,163,0.212349
3,4,1168,292,324,68,218,0.135571
4,5,1168,292,328,29,179,0.122542


CV RMSLE: 0.144820 ± 0.038047
OOF RMSLE: 0.148764
Submission: submissions/submission_BASE-20260719-TORCH-LINEAR-SUB-01.csv


In [5]:
run_timestamp_utc = datetime.now(timezone.utc).isoformat()
summary = {
    "experiment_id": EXPERIMENT_ID,
    "fold_scores": fold_results["rmsle"].tolist(),
    "cv_mean": cv_mean,
    "cv_std": cv_std,
    "oof_rmsle": oof_rmsle,
    "best_epochs": fold_results["best_epoch"].tolist(),
    "stopping_epochs": fold_results["stopping_epoch"].tolist(),
    "encoded_features_min": int(fold_results["encoded_features"].min()),
    "encoded_features_max": int(fold_results["encoded_features"].max()),
    "test_prediction_strategy": "mean of 5 fold log predictions, then expm1",
}
submission_validation = {
    "experiment_id": EXPERIMENT_ID,
    "submission_path": str(SUBMISSION_PATH.relative_to(ROOT)),
    "columns": submission.columns.tolist(),
    "rows": int(len(submission)),
    "dtypes": {column: str(dtype) for column, dtype in submission.dtypes.items()},
    "id_matches_test_order": bool(submission["Id"].equals(test["Id"])),
    "unique_ids": bool(submission["Id"].is_unique),
    "missing_predictions": int(submission["SalePrice"].isna().sum()),
    "nonfinite_predictions": int((~np.isfinite(submission["SalePrice"])).sum()),
    "nonpositive_predictions": int(submission["SalePrice"].le(0).sum()),
    "prediction_min": float(submission["SalePrice"].min()),
    "prediction_max": float(submission["SalePrice"].max()),
    "fold_prediction_count": int(test_log_prediction_matrix.shape[0]),
    "kaggle_public_score": KAGGLE_PUBLIC_SCORE,
    "kaggle_private_score": "unverified",
    "kaggle_score_source": "user_reported",
    "kaggle_score_recorded_at_utc": KAGGLE_SCORE_RECORDED_AT_UTC,
}
metrics = {
    "run_timestamp_utc": run_timestamp_utc,
    "source": {
        "path": "data/train.csv",
        "sha256": source_sha256,
        "rows": int(train.shape[0]),
        "columns": int(train.shape[1]),
        "predictor_count": 79,
        "test_csv_used": True,
        "test_path": "data/test.csv",
        "test_sha256": test_sha256,
        "test_rows": int(test.shape[0]),
        "sample_submission_path": "data/sample_submission.csv",
        "sample_submission_sha256": sample_submission_sha256,
    },
    "preprocessing": {
        "numeric": "fold median imputation and StandardScaler",
        "categorical": "structural absence labels, Unknown, fold one-hot encoding",
        "excluded_model_columns": ["Id", "SalePrice"],
        "generated_features": [],
    },
    "validation": {
        "splitter": "KFold",
        "n_splits": N_SPLITS,
        "shuffle": True,
        "random_state": SEED,
        "target": "log1p(SalePrice)",
        "metric": "RMSLE / RMSE on log1p target",
    },
    "model": {
        "class": "LinearHousePriceModel(nn.Module)",
        "architecture": "nn.Linear(input_dim, 1)",
        "hidden_layers": 0,
        "activation": None,
        "optimizer": "Adam",
        "learning_rate": LEARNING_RATE,
        "weight_decay": 0.0,
        "loss": "MSELoss on log1p(SalePrice)",
        "max_epochs": MAX_EPOCHS,
        "patience": PATIENCE,
        "device": str(DEVICE),
        "precision": "float32",
        "seed": SEED,
    },
    "experiments": {EXPERIMENT_ID: summary},
    "artifacts": {
        "notebook": "notebooks/house_prices_baseline.ipynb",
        "oof_predictions": "reports/baseline_oof_predictions.csv",
        "test_fold_predictions": "reports/baseline_test_fold_predictions.csv",
        "submission": str(SUBMISSION_PATH.relative_to(ROOT)),
        "submission_validation": "reports/baseline_submission_validation.json",
    },
    "submission_validation": submission_validation,
    "environment": {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scikit_learn": sklearn.__version__,
        "torch": torch.__version__,
    },
    "kaggle_scores": {
        "public": KAGGLE_PUBLIC_SCORE,
        "private": "unverified",
        "source": "user_reported",
        "recorded_at_utc": KAGGLE_SCORE_RECORDED_AT_UTC,
    },
}
METRICS_PATH.write_text(
    json.dumps(metrics, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
SUBMISSION_VALIDATION_PATH.write_text(
    json.dumps(submission_validation, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print(f"Saved: {METRICS_PATH.relative_to(ROOT)}")

Saved: reports/baseline_metrics.json
